# Diagnosis: Huge PDE pricing error for NVDA

**Symptom.** When we calibrate with the **PDE Solver** (`american_method="pde"`) and then price American options with the **PDE Solver method**, the average pricing error for NVDA is enormous.

**Conclusion (TL;DR).** The MCS ADI solver is advertised as *unconditionally stable* but it is not. Both the C++ kernel (`pricing/cpp/heston_mcs.cpp`) and the Python fallback (`pricing/heston_pde_american.py`) use the stability parameter

$$\theta = 1 - \tfrac{1}{\sqrt2} \approx 0.293,$$

which is **below the unconditional-stability threshold** for this scheme. The stiff variance-diffusion modes then have amplification factor $|g|>1$, so the solution **overflows** ($10^{12}$–$10^{60}$) once `dt` is large relative to $\sigma^2 v/dv^2$. This happens routinely for:
- high **vol-of-vol** $\sigma$ (NVDA calibrates near the $\sigma=3.0$ bound), and/or
- finer **v-grids** (`Nv`), which the calibration/pricing pages allow up to 200.

Setting $\theta = \tfrac12$ makes the scheme robustly stable (and keeps full 2nd-order accuracy). The fix is one line in each of the two files; it is **not applied here** — this notebook only demonstrates the diagnosis.

The instability poisons **both** stages: PDE pricing returns garbage, and PDE calibration's finite-difference gradient steps into the unstable region, corrupting the objective and producing bad parameters.

In [1]:
import os, sys
# Run from repo root so package imports resolve
if os.path.basename(os.getcwd()) == "utils":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd

from pricing.heston_pde_american import (
    _USE_CPP,
    heston_pde_american,            # dispatcher -> C++ if available
    _heston_pde_american_python,    # pure-Python reference
)
from pricing.european import heston_european_call_option, heston_european_put_option

print("Using C++ kernel:", _USE_CPP)

Using C++ kernel: True


## Step 1 — Sanity check at the default grid

An American call with no dividends must equal the European call (early exercise never optimal), and an American put must be $\ge$ its European value. At the default `40/20/40` grid with *moderate* $\sigma$ the solver looks plausible but already slightly **underprices** (coarse `Ns`: with `Smax = 4·max(S0,K)`, the spot `S0` lands on only node ~10 of 40).

In [2]:
S0, r, q, T = 130.0, 0.05, 0.0, 0.5
v0, kappa, theta, sigma, rho = 0.30, 2.0, 0.30, 1.2, -0.6

rows = []
for K in [110, 120, 130, 140, 160]:
    euc = heston_european_call_option(S0, K, r, T, v0, kappa, theta, sigma, rho, q)
    eup = heston_european_put_option(S0, K, r, T, v0, kappa, theta, sigma, rho, q)
    pdc = heston_pde_american(S0, K, r, q, T, v0, kappa, theta, sigma, rho, "call")
    pdp = heston_pde_american(S0, K, r, q, T, v0, kappa, theta, sigma, rho, "put")
    rows.append((K, euc, pdc, pdc - euc, eup, pdp, pdp - eup))
pd.DataFrame(rows, columns=["K", "call_EU", "call_PDE", "call_diff", "put_EU", "put_PDE", "put_diff"]).round(3)

,K,call_EU,call_PDE,call_diff,put_EU,put_PDE,put_diff
0,110,31.818,31.722,-0.097,9.103,9.019,-0.083
1,120,25.519,25.165,-0.354,12.556,12.316,-0.240
2,130,19.998,19.214,-0.784,16.789,16.277,-0.511
3,140,15.297,14.475,-0.822,21.840,21.552,-0.289
4,160,8.337,7.527,-0.810,34.386,34.788,0.402


## Step 2 — The smoking gun: refining the grid makes the price explode

A genuinely unconditionally stable scheme should **converge** as the grid is refined. Instead the solver diverges. Below, the put price blows up as `Nv` grows and as `σ` grows — at the **default** `40/20/40` grid it already overflows at `σ = 3.0`.

In [3]:
K = 130
grids = [(40, 20, 40), (80, 40, 80), (120, 60, 120), (200, 100, 200)]
out = []
for sig in [0.5, 1.0, 1.5, 2.0, 3.0]:
    p = (0.25, 1.5, 0.28, sig, -0.6)
    eu = heston_european_put_option(S0, K, r, T, *p, q)
    row = {"sigma": sig, "EU": round(eu, 3)}
    for (Ns, Nv, Nt) in grids:
        row[f"{Ns}/{Nv}/{Nt}"] = f"{heston_pde_american(S0, K, r, q, T, *p, 'put', Ns, Nv, Nt):.3e}"
    out.append(row)
pd.DataFrame(out)

,sigma,EU,40/20/40,80/40/80,120/60/120,200/100/200
0,0.5,16.398,1.649e+01,1.663e+01,1.666e+01,1.667e+01
1,1.0,15.635,1.534e+01,1.568e+01,1.575e+01,5.148e+02
2,1.5,14.613,1.333e+01,1.379e+01,5.258e+13,1.217e+62
3,2.0,13.540,1.145e+01,2.828e+12,2.040e+38,5.983e+96
4,3.0,11.631,1.065e+06,1.848e+32,3.563e+63,6.896e+128


## Step 3 — Isolate the mechanism

Is it the cross-derivative (correlation) term, or the time integration?
- With `rho = 0` (mixed term off) it still blows up identically → **not** the correlation term.
- The blow-up grows with `Nv` (smaller `dv`) and is only cured by making `dt` tiny (large `Nt`). That is the signature of a **conditionally** stable time scheme: explicit-Euler-like stability limit `dt < C·dv²/(σ²·vmax)`. The code only enforces `Nt ≥ 50·T` (i.e. `dt ≤ 0.02`), independent of `Nv` and `σ`.

In [4]:
# Use the pure-Python kernel so we can later rebuild it with a different theta.
p = (0.25, 1.5, 0.28, 2.0, -0.6)  # sigma = 2.0
print("rho = 0 vs rho = -0.6, varying Nv (Ns=40, Nt=40):")
for rho_ in [0.0, -0.6]:
    pr = (0.25, 1.5, 0.28, 2.0, rho_)
    vals = [f"Nv={Nv}:{_heston_pde_american_python(S0, K, r, q, T, *pr, 'put', 40, Nv, 40):.2e}" for Nv in [20, 40, 60, 100]]
    print(f"  rho={rho_:+.1f}  " + "  ".join(vals))

print("\nFix Ns=40, Nv=60; refine Nt only (smaller dt) -> stability returns:")
for Nt in [40, 100, 200, 400]:
    print(f"  Nt={Nt:3}: {_heston_pde_american_python(S0, K, r, q, T, *p, 'put', 40, 60, Nt):.3e}")

rho = 0 vs rho = -0.6, varying Nv (Ns=40, Nt=40):
  rho=+0.0  Nv=20:1.27e+01  Nv=40:1.22e+12  Nv=60:1.23e+19  Nv=100:3.96e+25


  rho=-0.6  Nv=20:1.14e+01  Nv=40:3.99e+12  Nv=60:9.34e+17  Nv=100:1.30e+21

Fix Ns=40, Nv=60; refine Nt only (smaller dt) -> stability returns:
  Nt= 40: 9.337e+17
  Nt=100: 5.216e+35


  Nt=200: 5.040e+32


  Nt=400: 1.160e+01


## Step 4 — Root cause: the MCS stability parameter $\theta$

Both kernels hard-code `th = 1 - 1/sqrt(2) ≈ 0.293`:
- `pricing/cpp/heston_mcs.cpp:243`
- `pricing/heston_pde_american.py` (`_heston_pde_american_python`)

For the Douglas/MCS family, the stiff-mode amplification factor tends to $-(1-\theta)/\theta$ as $dt\,\lambda \to -\infty$; that exceeds 1 in magnitude when $\theta < \tfrac12$ for the bare directional solves. The MCS corrector relaxes this somewhat but, in **this** implementation (central-differenced variance convection + degenerate $v=0$ boundary), even $\theta = 1/3$ is not enough. We rebuild the Python kernel with several $\theta$ values and re-run the divergence test.

In [5]:
import inspect
import pricing.heston_pde_american as m

_src = inspect.getsource(m._heston_pde_american_python)

def make_solver(theta_value):
    """Recompile the Python kernel with a chosen MCS theta (no repo edits)."""
    s = _src.replace("th = 1.0 - 1.0 / np.sqrt(2.0)", f"th = {theta_value}")
    ns = {"np": np}
    exec(s, ns)
    return ns["_heston_pde_american_python"]

p = (0.25, 1.5, 0.28, 2.0, -0.6)  # sigma = 2.0
eu = heston_european_put_option(S0, K, r, T, *p, q)
print(f"EU put ref = {eu:.3f}\n")
for label, th in [("1-1/sqrt2 (current)", 1.0 - 1.0/np.sqrt(2)), ("1/3", 1/3), ("1/2", 0.5), ("2/3", 2/3)]:
    f = make_solver(th)
    vals = [f"Nv={Nv}:{f(S0, K, r, q, T, *p, 'put', 40, Nv, 40):.3e}" for Nv in [20, 40, 60, 100]]
    print(f"  theta={label:20s}  " + "  ".join(vals))

EU put ref = 13.540

  theta=1-1/sqrt2 (current)   Nv=20:1.145e+01  Nv=40:3.992e+12  Nv=60:9.337e+17  Nv=100:1.304e+21


  theta=1/3                   Nv=20:1.145e+01  Nv=40:6.095e+04  Nv=60:3.060e+12  Nv=100:2.094e+18
  theta=1/2                   Nv=20:1.145e+01  Nv=40:1.155e+01  Nv=60:1.158e+01  Nv=100:1.160e+01


  theta=2/3                   Nv=20:1.145e+01  Nv=40:1.155e+01  Nv=60:1.159e+01  Nv=100:1.162e+01


**Result.** `theta = 1 - 1/sqrt2` and `theta = 1/3` both diverge; **`theta = 1/2` (and `2/3`) are stable** across every `Nv`.

## Step 5 — With $\theta = 1/2$ the scheme converges *and* respects no-arbitrage

Under the stable $\theta$, refining the grid converges; the American put correctly sits **above** the European put (early-exercise premium), and the no-dividend American call lands within grid-discretization tolerance of the European call.

In [6]:
f = make_solver(0.5)
p = (0.25, 1.5, 0.28, 1.0, -0.6)  # sigma = 1.0
euP = heston_european_put_option(S0, K, r, T, *p, q)
euC = heston_european_call_option(S0, K, r, T, *p, q)
print(f"EU put={euP:.3f}  EU call={euC:.3f}  (American must be >= these)")
for (Ns, Nv, Nt) in [(40, 20, 40), (80, 40, 80), (160, 80, 160), (320, 160, 320)]:
    vp = f(S0, K, r, q, T, *p, "put", Ns, Nv, Nt)
    vc = f(S0, K, r, q, T, *p, "call", Ns, Nv, Nt)
    print(f"  {Ns:3}/{Nv:3}/{Nt:3}:  put={vp:.3f}  call={vc:.3f}")

EU put=15.635  EU call=18.845  (American must be >= these)
   40/ 20/ 40:  put=15.344  call=18.302
   80/ 40/ 80:  put=15.677  call=18.619


  160/ 80/160:  put=15.776  call=18.709


  320/160/320:  put=15.816  call=18.744


## Step 6 — End-to-end effect on calibration + pricing

Build a small self-consistent NVDA-like chain (European "market" prices from known params), calibrate with `american_method="pde"`, then price with the PDE method and measure the error. With the current $\theta$, any FD step into the unstable region corrupts the objective; the effect is most violent when `σ` is high or the grid is fine.

In [7]:
from services.calibration_service import calibrate_option_chain
from services.pricing_service import price_option_frame

true = (0.25, 1.5, 0.28, 1.0, -0.6)
mats = {0.08: "2026-07-18", 0.25: "2026-09-18", 0.5: "2026-12-18", 1.0: "2027-06-18"}
rows = []
for Tn, mat in mats.items():
    for Kn in [110, 120, 125, 130, 135, 140, 150]:
        cp = heston_european_call_option(S0, Kn, r, Tn, *true, q)
        pp = heston_european_put_option(S0, Kn, r, Tn, *true, q)
        for typ, px in (("call", cp), ("put", pp)):
            rows.append(dict(ticker="NVDA", type=typ, maturity=mat, strike=Kn, spot=S0,
                             bid=px*0.99, ask=px*1.01, volume=100, openInterest=100,
                             T=Tn, ExerciseStyle="american", q=q, r=r))
chain = pd.DataFrame(rows)

for method in ["european_proxy", "pde"]:
    res, _ = calibrate_option_chain(chain, r=r, q=q, american_method=method,
                                    max_expiries=4, contracts_per_expiry=7)
    price_method = "auto" if method == "european_proxy" else "pde"
    mp = price_option_frame(chain, r=r, q=q, heston_params=res.params, american_method=price_method)
    mid = (chain["bid"] + chain["ask"]) / 2
    ape = (mp - mid).abs() / mid
    print(f"== {method} ==  params={tuple(round(x,3) for x in res.params.as_tuple())}  loss={res.loss:.4f}")
    print(f"   mean |pct err| = {ape.mean()*100:5.2f}%   max |pct err| = {ape.max()*100:5.1f}%\n")

[calibration] using data-driven guess/bounds (4 liquid maturities): v0=0.2444 kappa=1.0870 theta=0.2223 sigma=0.6220 rho=-0.9441


== european_proxy ==  params=(0.25, 1.5, 0.28, 1.0, -0.6)  loss=0.0000
   mean |pct err| =  1.23%   max |pct err| =   8.7%

[calibration] using data-driven guess/bounds (4 liquid maturities): v0=0.2444 kappa=1.0870 theta=0.2223 sigma=0.6220 rho=-0.9441


== pde ==  params=(0.251, 2.288, 0.252, 0.862, -0.821)  loss=0.2997
   mean |pct err| =  1.20%   max |pct err| =  11.1%



> Note: this small, self-consistent chain keeps the calibrated `σ` modest, so the catastrophic overflow does not fully show here. On the real NVDA chain `σ` calibrates near the 3.0 bound (and the app allows finer `Nv`), which is exactly the regime that overflows at Step 2 — that is what produces the "huge" average pricing error in production.

## Proposed fix (not applied)

Change the MCS stability parameter from `1 - 1/sqrt(2)` to `0.5` in **both** kernels, then rebuild the C++ extension:

| File | Line | From | To |
|---|---|---|---|
| `pricing/cpp/heston_mcs.cpp` | 243 | `const double th = 1.0 - 1.0 / std::sqrt(2.0);` | `const double th = 0.5;` |
| `pricing/heston_pde_american.py` | ~36 | `th = 1.0 - 1.0 / np.sqrt(2.0)` | `th = 0.5` |

```bash
python setup_cpp.py build_ext --inplace   # rebuild C++ so the change takes effect in pricing
```

**Secondary (smaller) accuracy items**, separate from the stability bug:
- Coarse `Ns` (default 40, with `Smax = 4·max(S0,K)`) puts `S0` on ~node 10 → a few-percent underprice. Convergence improves with finer `Ns`.
- `vmax = clip(5·v0, 0.5, 3.0)` can truncate the variance domain for very high `σ`, mildly underpricing long-dated/high-vol contracts.
- `-ffast-math` in `setup_cpp.py` disables NaN/Inf guarding; worth reconsidering once the scheme is stable.